# A gamified LEGO framework for teaching signals and systems
## Statistical Analysis
**Andrés Quintero-Zea · Universidad EIA · Interpersona: an International Journal**

This notebook reproduces all the analysis of the article using specialized libraries for
statistics:

| Library | Version | Main use |
| --- | :---: | --- |
| `dabest` | 2025.10.20 | Estimation statistics (Gardner-Altman, Cumming) |
| `matplotlib` | 3.10.0 | Publication Figures |
| `numpy` | 2.1.3 | High-performance, multi-dimensional array and matrix calculations |
| `pandas` | 2.2.3 | Data manipulation and analysis |
| `pingouin` | 0.6.1 | Parametric and non-parametric tests, effect sizes, RM-ANOVA, reliability |
| `scikit-learn` | 1.9.0 | Stardardize predictors for multiple linear regression |
| `scipy` | 1.15.2 | Distributions, Shapiro-Wilk, Friedman, correlations |
| `seaborn` | 0.13.2 | Publication Figures |
| `statannotations` | 0.7.2 | Add statistical annotations on boxplots or barplots generated with seaborn |
| `statsmodels` | 0.14.6 | OLS with full inference, mediation, diagnostics |

**Installation:**
```bash
conda env create -f environment.yml
conda activate gamified-lego-signals-systems
jupyter lab
```

**Dataset:** the notebook expects `data/raw/data_share.xlsx` in the repository.

**Reproducibility:** seed `SEED = 2025`

## §0 · Imports, constants and configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import importlib.metadata

# ── Standard libraries ──────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path

# ── Scipy ───────────────────────────────────────────────────
from scipy import stats
from scipy.stats import (shapiro, normaltest, ttest_rel, wilcoxon,
                         friedmanchisquare, mannwhitneyu,
                         pearsonr, spearmanr, gaussian_kde)

# ── Pingouin ─────────────────────── ────────────────────────
import pingouin as pg

# State models
import statsmodels
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.mediation import Mediation

# ── Visualization ───────────────────── ──────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import dabest
from statannotations.Annotator import Annotator
from scipy.stats import probplot as sp_probplot

# ── scikit-learn and statmodels ─────────────────────────────
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── Reproducibility ──────────────────── ────────────────────
SEED   = 2025
N_BOOT = 5_000
RNG    = np.random.default_rng(SEED)

# ── Routes ───────────────────────── ──────────────────────────
# Resolve the repository root whether Jupyter is launched from the root or notebooks/.
CWD = Path.cwd()
ROOT_CANDIDATES = [CWD, CWD.parent]
ROOT_DIR = next(
    (p for p in ROOT_CANDIDATES if (p / 'data' / 'raw' / 'data_share.xlsx').is_file()),
    None
)
if ROOT_DIR is None:
    checked = [str(p / 'data' / 'raw' / 'data_share.xlsx') for p in ROOT_CANDIDATES]
    raise FileNotFoundError(
        "Dataset not found. Expected data/raw/data_share.xlsx. "
        f"Checked: {checked}"
    )

DATA_PATH = ROOT_DIR / 'data' / 'raw' / 'data_share.xlsx'
FIG_DIR   = ROOT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Domain constants
MAX_SCORE  = 5.0
ITEMS      = ['SR', 'TS', 'TR', 'TSC', 'CONV']
ITEM_NAMES = {'SR':'Signal Representation','TS':'Time Shifting',
              'TR':'Time Reversal','TSC':'Time Scaling','CONV':'Convolution'}
ITEM_LABELS= ['Signal\nRepresentation','Time\nShifting','Time\nReversal',
              'Time\nScaling','Convolution']
Q_COLS     = ['Q1','Q2','Q3','Q4','Q5','Q6']
Q_TEXT     = {'Q1':'The activities increased my engagement',
              'Q2':'The challenges were motivating',
              'Q3':'The badges encouraged participation',
              'Q4':'The activities improved my understanding',
              'Q5':'Collaboration enhanced my learning',
              'Q6':'I would recommend this methodology'}
Q_SHORT    = ['Increased\nengagement (Q1)','Motivating\nchallenges (Q2)',
              'Badges encouraged\nparticipation (Q3)',
              'Improved\nunderstanding (Q4)',
              'Collaboration enhanced\nlearning (Q5)',
              'Would recommend\nmethodology (Q6)']
BADGES     = ['SignalTransformer','ConvolutionMaster','Explorer']

# ── Palette ───────────────────────── ─────────────────────────
C_PRE    = '#4C72B0'
C_POST   = '#DD8452'
C_HAKE   = '#555555'
C_ACCENT = '#888888'

# ── Global style ───────────────────── ──────────────────────
plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':11,
    'axes.titlesize':12,'axes.labelsize':11,
    'axes.spines.top':False,'axes.spines.right':False,
    'figure.dpi':300,'savefig.dpi':600,
    'savefig.bbox':'tight','savefig.facecolor':'white',
})
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Helper ──────────────────────────────────────────────────
def fmt_p(p):
    return '< .001' if p < .001 else f'= {p:.3f}'

def stars(p):
    return '****' if p<0.0001 else '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'ns'

print('§0 OK — all libraries loaded.')
print(f'  dabest          : {importlib.metadata.version("dabest")}')
print(f'  matplotlib      : {importlib.metadata.version("matplotlib")}')
print(f'  numpy           : {importlib.metadata.version("numpy")}')
print(f'  pandas          : {importlib.metadata.version("pandas")}')
print(f'  pingouin        : {importlib.metadata.version("pingouin")}')
print(f'  scikit-learn    : {importlib.metadata.version("scikit-learn")}')
print(f'  scipy           : {importlib.metadata.version("scipy")}')
print(f'  seaborn         : {importlib.metadata.version("seaborn")}')
print(f'  statannotations : {importlib.metadata.version("statannotations")}')
print(f'  statsmodels     : {importlib.metadata.version("statsmodels")}')

## §1 · Data loading and validation

In [ ]:
df = pd.read_excel(DATA_PATH)
for b in BADGES:
    df[b] = df[b].astype(bool)

# ── Derived variables ─────────────────────────────────────
df['Gain']         = df['POST'] - df['PRE']
df['Hake_g']       = (df['POST'] - df['PRE']) / (MAX_SCORE - df['PRE'])
for it in ITEMS:
    df[f'{it}_Gain']   = df[f'{it}_POST'] - df[f'{it}_PRE']
    df[f'{it}_Hake_g'] = (df[f'{it}_POST'] - df[f'{it}_PRE']) / (MAX_SCORE - df[f'{it}_PRE'])
df['Likert_mean']  = df[Q_COLS].mean(axis=1)
df['Badges_total'] = df[BADGES].sum(axis=1)

# ── Basic validation ───────────────────────────────────────
print(f'Estudiantes : {len(df)}')
print(f'Grupos      : {df["grupo"].nunique()} '
      f'(tamaños: {sorted(df.groupby("grupo").size().unique().tolist())})')
print(f'Valores NaN : {df.isna().sum().sum()}')
pre_ok  = np.allclose(df[[f'{it}_PRE'  for it in ITEMS]].mean(axis=1), df['PRE'],  atol=.011)
post_ok = np.allclose(df[[f'{it}_POST' for it in ITEMS]].mean(axis=1), df['POST'], atol=.011)
print(f'Consistencia PRE/POST con ítems: {pre_ok} / {post_ok}')
df[['ID','grupo','PRE','POST','Gain','Hake_g','Likert_mean','Badges_total']].head()

## §2 · Descriptive statistics

In [ ]:
# ── Global and item descriptives ────────────────────────
pre_cols  = [f'{it}_PRE'  for it in ITEMS] + ['PRE']
post_cols = [f'{it}_POST' for it in ITEMS] + ['POST']

rows = []
for it_lbl, pre_c, post_c in zip(
    [ITEM_NAMES[it] for it in ITEMS] + ['GLOBAL'], pre_cols, post_cols
):
    for phase, col in [('PRE', pre_c), ('POST', post_c)]:
        s = df[col]
        ci = pg.compute_bootci(s.values, func='mean', n_boot=N_BOOT,
                               confidence=.95, seed=SEED)
        rows.append({
            'Item': it_lbl, 'Phase': phase,
            'N': len(s), 'M': s.mean(), 'SD': s.std(ddof=1),
            'Median': s.median(), 'IQR': s.quantile(.75)-s.quantile(.25),
            'CI95_lo': ci[0], 'CI95_hi': ci[1],
            'Skew': stats.skew(s), 'Kurt': stats.kurtosis(s),
        })

desc_df = pd.DataFrame(rows)
print('Table 1 — PRE and POST descriptive statistics by item')
desc_df

## §3 · Assumptions: normality

In [ ]:
# ── Normality table ─────────────────────────────────────
norm_vars = (['Gain','Hake_g']
             + [f'{it}_Gain' for it in ITEMS]
             + ['PRE','POST'])

norm_rows = []
for col in norm_vars:
    W_sw, p_sw = shapiro(df[col])
    k2, p_k2   = normaltest(df[col])
    norm_rows.append({
        'Variable': col,
        'Shapiro_W': W_sw, 'Shapiro_p': p_sw,
        'K2': k2,          'K2_p': p_k2,
        'Normal': 'Yes' if p_sw >= .05 else 'No',
    })

norm_df = pd.DataFrame(norm_rows)
print(norm_df.to_string(index=False))

# Decision reported dynamically from the table itself,
# instead of a fixed text that could become out of sync if the
# data changed (e.g. between versions of the dataset).
gain_row   = norm_df.loc[norm_df['Variable'] == 'Gain'].iloc[0]
all_normal = (norm_df['Normal'] == 'Yes').all()
if gain_row['Normal'] == 'Yes':
    print(f"\nDecision: the overall difference (Gain) is normally compatible."
    f"\n(Shapiro W={gain_row['Shapiro_W']:.3f}, p={gain_row['Shapiro_p']:.3f})"
    f"\nt and Wilcoxon tests are reported in parallel as a robustness check.")
else:
    print(f"\nDecision: Non-normal distribution in the overall difference (Gain)"
          f"\n(Shapiro W={gain_row['Shapiro_W']:.3f}, p={gain_row['Shapiro_p']:.3f})"
          f"\n→ Wilcoxon is reported as the main test, with t in parallel.")
if not all_normal:
    print("\nNote: At least one other variable in the table above also deviates from normality;"
          "\nReview each variable individually before interpreting parametric tests by item.")

In [ ]:
# ── Q-Q plots con scipy + seaborn ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in zip(axes,
                          ['PRE', 'POST', 'Gain'],
                          ['Pre-test scores', 'Post-test scores',
                           'Paired difference (POST-PRE)']):
    (osm, osr), (slope, intercept, r) = sp_probplot(df[col], dist='norm')
    sns.scatterplot(x=osm, y=osr, ax=ax, color=C_PRE, alpha=0.8, s=35, zorder=3)
    line_x = np.array([osm[0], osm[-1]])
    ax.plot(line_x, slope * line_x + intercept, color=C_ACCENT, lw=1.5,
            label=f'$r = {r:.3f}$')
    ax.set_title(title)
    ax.set_xlabel('Theoretical quantiles')
    ax.set_ylabel('Sample quantiles')
    ax.legend(frameon=False, fontsize=9)
fig.suptitle('Q-Q plots — verificación de normalidad', fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_qqplots.png')
fig.savefig(FIG_DIR / 'fig_qqplots.pdf')
plt.show()

## §4 · Hypothesis testing

In [ ]:
# ── Paired t test + Wilcoxon per item ────────────────────
results = []
for label, pre_c, post_c in (
    [('Global','PRE','POST')]
    + [(ITEM_NAMES[it], f'{it}_PRE', f'{it}_POST') for it in ITEMS]
):
    pre, post = df[pre_c], df[post_c]

    # pingouin paired t-test (includes d, power, CI of effect)
    tt = pg.ttest(pre, post, paired=True, correction=False)

    # Wilcoxon pingouin (includes CLES and RBC)
    wx = pg.wilcoxon(pre, post, alternative='two-sided')

    results.append({
        'Item'      : label,
        'M_pre'     : pre.mean(),
        'M_post'    : post.mean(),
        't'         : tt['T'].iloc[0],
        'df_t'      : tt['dof'].iloc[0],
        'p_t'       : tt['p_val'].iloc[0],
        'CI95_d_lo' : tt['CI95'].iloc[0][0],
        'CI95_d_hi' : tt['CI95'].iloc[0][1],
        'Cohen_dz'  : tt['cohen_d'].iloc[0],
        'Hedges_g'  : pg.compute_effsize(post, pre, paired=True, eftype='hedges'),
        'Power'     : tt['power'].iloc[0],
        'W_wilcoxon': wx['W_val'].iloc[0],
        'p_W'       : wx['p_val'].iloc[0],
        'CLES'      : wx['CLES'].iloc[0],
        'RBC'       : wx['RBC'].iloc[0],
        'sig'       : stars(tt['p_val'].iloc[0]),
    })

ht_df = pd.DataFrame(results)
print('Table 2 — Hypothesis tests and effect sizes')
ht_df

## §5 Normalized Hake gain

In [ ]:
# ── g per item ─────────────────────── ───────────────────────
hake_rows = []
for label, g_col, pre_c, post_c in (
    [('Global','Hake_g','PRE','POST')]
    + [(ITEM_NAMES[it],f'{it}_Hake_g',f'{it}_PRE',f'{it}_POST') for it in ITEMS]
):
    gvals    = df[g_col].values
    g_single = (df[post_c].mean() - df[pre_c].mean()) / (MAX_SCORE - df[pre_c].mean())
    ci       = pg.compute_bootci(gvals, func='mean', n_boot=N_BOOT,
                                 confidence=.95, seed=SEED)
    hake_rows.append({
        'Item'    : label,
        'g_single': g_single,
        'g_mean'  : gvals.mean(),
        'g_SD'    : gvals.std(ddof=1),
        'CI95_lo' : ci[0], 'CI95_hi': ci[1],
        'Cat'     : 'High' if gvals.mean()>=.7 else
                    'Medium' if gvals.mean()>=.3 else 'Low',
    })

hake_df = pd.DataFrame(hake_rows)
print('Normalized Hake gain per item:')
print(hake_df.to_string(index=False))

In [ ]:
# ── Friedman between topics ─────────────────────────
# Reshape long format to pingouin
g_long = df[['ID'] + [f'{it}_Hake_g' for it in ITEMS]].copy()
g_long = g_long.melt(id_vars='ID', var_name='Item', value_name='Hake_g')
g_long['Item'] = g_long['Item'].map({f'{it}_Hake_g': ITEM_NAMES[it] for it in ITEMS})

friedman_res = pg.friedman(data=g_long, dv='Hake_g', within='Item', subject='ID')
print('Friedman test — differences between subjects:')
print(friedman_res.to_string())

# Post-hoc if p < .05
pf_val = friedman_res['p_unc'].iloc[0]
if pf_val < .05:
    print('\nPost-hoc pairwise Wilcoxon (Holm correction):')
    ph = pg.pairwise_tests(data=g_long, dv='Hake_g', within='Item',
                           subject='ID', parametric=False,
                           padjust='holm')
    print(ph[['A','B','W_val','p_unc','p_corr','p_adjust','hedges']].to_string(index=False))
else:
    print(f'\np = {fmt_p(pf_val)} → no significant differences between topics.')
    print('No post-hoc comparisons are required.')

## §6 · Instrument reliability

In [ ]:
# ── Reliability of the conceptual instrument (PRE and POST items) ──
PRE_ITEMS  = [f'{it}_PRE'  for it in ITEMS]   # SR_PRE, TS_PRE, TR_PRE, TSC_PRE, CONV_PRE
POST_ITEMS = [f'{it}_POST' for it in ITEMS]   # SR_POST, TS_POST, TR_POST, TSC_POST, CONV_POST

def interpret_alpha(a):
    return ('Excellent' if a>=.9 else 'Good' if a>=.8
            else 'Acceptable' if a>=.7 else 'Questionable')

for label, cols in [('PRE', PRE_ITEMS), ('POST', POST_ITEMS)]:
    alpha, ci = pg.cronbach_alpha(df[cols])
    print(f'α {label} = {alpha:.3f}  95% CI [{ci[0]:.3f}, {ci[1]:.3f}]  '
          f'({interpret_alpha(alpha)})')

    print(f'  "If item removed" analysis ({label}):')
    for col in cols:
        rest        = df[cols].drop(columns=col).sum(axis=1)
        r_it, p_it  = pearsonr(df[col], rest)
        a_drop, _   = pg.cronbach_alpha(df[cols].drop(columns=col))
        print(f'    {col}: item-total r = {r_it:.3f} (p {fmt_p(p_it)}), '
              f'α if removed = {a_drop:.3f}')
    print()

In [ ]:
# ── Cronbach's alpha (single calculation, reused in §10 and §16) ─
alpha_likert, ci_likert = pg.cronbach_alpha(df[Q_COLS])
print(f'Cronbach α (Likert Q1-Q6) = {alpha_likert:.3f}  '
      f'95% CI [{ci_likert[0]:.3f}, {ci_likert[1]:.3f}]')
interp = ('Excelent' if alpha_likert>=.9 else 'Good' if alpha_likert>=.8
          else 'Aceptable' if alpha_likert>=.7 else 'Questionable')
print(f'Interpretación: {interp}')

# ── Corrected item-total correlations ─────────────────────
print('\nCorrected item-total correlations:')
item_total_rows = []
for q in Q_COLS:
    rest   = df[Q_COLS].drop(columns=q).sum(axis=1)
    r, p   = pearsonr(df[q], rest)
    a_drop, _ = pg.cronbach_alpha(df[Q_COLS].drop(columns=q))
    item_total_rows.append({'Item': q, 'r_it': r, 'p': p, 'alpha_if_removed': a_drop})
    print(f'  {q}: r_it = {r:.3f} (p {fmt_p(p)}), '
          f'α if removed = {a_drop:.3f}')
item_total_df = pd.DataFrame(item_total_rows)


In [ ]:
# ICC for Likert scale
# Reshape: each Q is a 'rater', each student is a 'target'
icc_df = df[Q_COLS].reset_index().melt(id_vars='index',
                                        var_name='Rater', value_name='Score')
icc_df = icc_df.rename(columns={'index':'Target'})
icc = pg.intraclass_corr(data=icc_df, targets='Target',
                          raters='Rater', ratings='Score')
print('ICC — consistency between Likert items:')
icc

## §7 · PRE vs POST by item

In [ ]:
# --- Parameters ---
x_label       = 'Conceptual item'
y_label       = 'Score (0–5)'
alpha_bar     = 0.85
alpha_pts     = 0.50
label_fontsize = 14
marker_size   = 3
line_width    = 1.2
jitter_width  = 0.1
y_limit_max   = 5.1
sig_offset    = 0.18

# === DATA IN LONG FORMAT ===
records = []
for it, label in zip(ITEMS + [None], ITEM_LABELS + ['Global']):
    pre_col  = f'{it}_PRE'  if it else 'PRE'
    post_col = f'{it}_POST' if it else 'POST'
    for val in df[pre_col]:
        records.append({'Item': label, 'Phase': 'Pre-test', 'Score': val})
    for val in df[post_col]:
        records.append({'Item': label, 'Phase': 'Post-test', 'Score': val})

df_plot    = pd.DataFrame(records)
item_order = ITEM_LABELS + ['Global']

# === CREATE FIGURE ===
fig, ax = plt.subplots(figsize=(12, 7))

sns.barplot(data=df_plot, x='Item', y='Score', hue='Phase',
            palette={'Pre-test': C_PRE, 'Post-test': C_POST},
            alpha=alpha_bar, capsize=0.08,
            errorbar='sd', err_kws={'linewidth': line_width * 2},
            order=item_order, ax=ax)

sns.stripplot(data=df_plot, x='Item', y='Score', hue='Phase',
              palette={'Pre-test': C_PRE, 'Post-test': C_POST},
              alpha=alpha_pts, size=marker_size * 1.5, jitter=jitter_width,
              linewidth=1.0,edgecolor="black",
              order=item_order, dodge=True, ax=ax, legend=False)

# === DECORATIONS ===
pre_m  = [df[f'{it}_PRE'].mean()      if it else df['PRE'].mean()      for it in ITEMS + [None]]
post_m = [df[f'{it}_POST'].mean()     if it else df['POST'].mean()     for it in ITEMS + [None]]
post_s = [df[f'{it}_POST'].std(ddof=1) if it else df['POST'].std(ddof=1) for it in ITEMS + [None]]

ax.axvline(len(ITEMS) - 0.5, color='#808080', ls='--', lw=0.9, alpha=0.5)
ax.set_xlabel(x_label, fontsize=label_fontsize)
ax.set_ylabel(y_label, fontsize=label_fontsize)
ax.set_ylim(0, y_limit_max)
ax.set_xticklabels(item_order, ha='center', fontsize=label_fontsize)

# Clean duplicate legend (barplot + stripplot)
handles, labels_leg = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels_leg[:2], frameon=False,
          loc='upper left', bbox_to_anchor=(1.03, 1), fontsize=label_fontsize)

pairs = [
    ((label, "Pre-test"), (label, "Post-test"))
    for label in item_order
]

annotator = Annotator(
    ax,
    pairs,
    data=df_plot,
    x='Item',
    y='Score',
    hue='Phase',
    order=item_order
)

annotator.configure(
    test='Wilcoxon',
    text_format='star',
    loc='outside',
    comparisons_correction='holm',
    line_width=1.2,
    line_height=0.02,
    fontsize=12,
    hide_non_significant=False,
    verbose = 0
)

annotation_results = annotator.apply_and_annotate()
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_prepost.png')
fig.savefig(FIG_DIR / 'fig_prepost.pdf')
plt.show()
print('fig_prepost.png guardada.')


## §8 · Estimation plot (DABEST Gardner-Altman)

In [ ]:
# ── Format long for dabest ────────────────────────────────────
df_long = pd.DataFrame({
    'Score' : pd.concat([df['PRE'], df['POST']], ignore_index=True),
    'Phase' : ['Pre-test']*len(df) + ['Post-test']*len(df),
    'ID'    : pd.concat([df['ID'], df['ID']], ignore_index=True),
    'Group' : pd.concat([df['grupo'], df['grupo']], ignore_index=True),
})

# ── Convert columns to appropriate data types ──────────────
df_long['Phase'] = df_long['Phase'].astype(str)  # Ensure Phase is string type
df_long['ID'] = df_long['ID'].astype(str)        # Ensure ID is string type
df_long['Score'] = pd.to_numeric(df_long['Score'], errors='coerce')  # Ensure Score is numeric

# ── Remove any rows with NaN values ────────────────────────
df_long = df_long.dropna()

# ── dabest: paired mean difference (Gardner-Altman) ─────────
db_paired = dabest.load(
    df_long, idx=('Pre-test','Post-test'),
    x='Phase', y='Score', id_col='ID', paired='baseline'
)

# Estimation plot
fig = db_paired.mean_diff.plot(
    raw_marker_size=5,
    swarm_side='Competence score (0–5)',
    contrast_label='',
    color_col='Group',
    raw_desat=1,
    contrast_desat=1,
    raw_ylim=(0, 5),
    contrast_ylim=(0, 2),
    fig_size=(6, 4)
)

rawdata_axes = fig.axes[0]
effsize_axes = fig.axes[1]

rawdata_axes.set_xticklabels(["Pre-test","Post-test"])
effsize_axes.set_xticklabels(["Gain","Gain"])
effsize_axes.get_legend().remove()
fig.savefig(FIG_DIR / 'fig_dabest_global.png')
fig.savefig(FIG_DIR / 'fig_dabest_global.pdf')
plt.show()

# Numerical bootstrap results
print('\nEstimation statistics (dabest):')
db_paired.mean_diff.statistical_tests

In [ ]:
# ── dabest: Cumming plot — all items at once ───────
# Allows you to visually compare the 5 + global items in a single panel
pairs = [(f'{it}_PRE', f'{it}_POST') for it in ITEMS] + [('PRE','POST')]
labels_pairs = [ITEM_NAMES[it] for it in ITEMS] + ['Global']
labels = [item.replace(' ', '\n') for item in labels_pairs]
new_labels = [
    x
    for label in labels
    for x in (' ', label + '\nGain')
]

# Reshape: a 'Condition' and a 'Score' column
all_cols = [c for pair in pairs for c in pair]
df_all   = df[all_cols + ['ID'] + ['grupo']].copy()
df_melt  = df_all.melt(id_vars=['ID','grupo'], var_name='Condition', value_name='Score')

db_all = dabest.load(
    df_melt, idx=pairs,
    x='Condition', y='Score', id_col='ID', paired='baseline'
)

fig2 = db_all.mean_diff.plot(
    raw_marker_size=1,
    contrast_label='Mean difference\n(Post – Pre)',
    color_col='grupo',
    delta_dot=True,
    contrast_marker_size=2
)
rawdata_axes = fig2.axes[0]
effsize_axes = fig2.axes[1]

rawdata_axes.set_xticklabels([item.replace('_', '\n') for item in all_cols])
effsize_axes.set_xticklabels(new_labels)
rawdata_axes.get_legend().set_visible(False)
effsize_axes.get_legend().set_visible(False)

fig2.savefig(FIG_DIR / 'fig_dabest_items.png')
fig2.savefig(FIG_DIR / 'fig_dabest_items.pdf')
plt.show()
print('fig_dabest_items.png guardada.')
db_all.mean_diff.statistical_tests

## §9 · Normalized Hake gains.

In [ ]:
# Reuse hake_df (calculated only once in §5) instead of
# rerun the bootstrap of 5,000 replicates for each variable.
labels_x  = ITEM_LABELS + ['Global']
order_map = [ITEM_NAMES[it] for it in ITEMS] + ['Global']
hake_plot = hake_df.set_index('Item').loc[order_map].reset_index()

hake_m  = hake_plot['g_mean'].to_numpy()
hake_lo = hake_plot['CI95_lo'].to_numpy()
hake_hi = hake_plot['CI95_hi'].to_numpy()
err_lo  = hake_m - hake_lo
err_hi  = hake_hi - hake_m
x_pos   = np.arange(len(labels_x))

# Long format for seaborn (from the *_Hake_g / Hake_g columns already existing in df)
records_h = []
for it, label in zip(ITEMS + [None], labels_x):
    g_col = f'{it}_Hake_g' if it else 'Hake_g'
    for val in df[g_col]:
        records_h.append({'Item': label, 'Hake_g': val,
                          'Type': 'Global' if it is None else 'Item'})
df_hake = pd.DataFrame(records_h)

fig, ax = plt.subplots(figsize=(9, 4.5))

# Palette: item = C_HAKE, global = C_ACCENT
palette_h = {lbl: (C_ACCENT if lbl == 'Global' else C_HAKE) for lbl in labels_x}
sns.barplot(data=df_hake, x='Item', y='Hake_g', order=labels_x,
            palette=palette_h, alpha=0.85, errorbar=None, ax=ax)

# Bootstrap error bars (reused from hake_df, §5)
ax.errorbar(x_pos, hake_m, yerr=[err_lo, err_hi], fmt='none',
            color='#000', capsize=5, linewidth=1.4, capthick=1.4)

# Zone Hake
ax.axhspan(0,  .3, color='red',   alpha=.06, zorder=0)
ax.axhspan(.3, .7, color='gold',  alpha=.07, zorder=0)
ax.axhspan(.7, 1., color='green', alpha=.07, zorder=0)
ax.axhline(.3, color='red',   ls='--', lw=.9, alpha=.5)
ax.axhline(.7, color='green', ls='--', lw=.9, alpha=.5)

for xi, m, hi in zip(x_pos, hake_m, hake_hi):
    ax.text(xi, hi + .015, f'{m:.3f}',
            ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax.axvline(len(ITEMS) - 0.5, color='gray', ls='--', lw=.8, alpha=.5)
ax.set_xticks(x_pos); ax.set_xticklabels(labels_x, ha='center')
ax.set_ylim(0, 1.05); ax.set_ylabel('Normalized gain  g  (Hake)')
ax.legend(handles=[
    mpatches.Patch(color='red',   alpha=.2, label='Low  (g < 0.3)'),
    mpatches.Patch(color='gold',  alpha=.2, label='Medium  (0.3 ≤ g < 0.7)'),
    mpatches.Patch(color='green', alpha=.2, label='High  (g ≥ 0.7)'),
], frameon=False, fontsize=9, loc='upper center', bbox_to_anchor=(0.5, 1.05),ncol=3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_hake.png')
fig.savefig(FIG_DIR / 'fig_hake.pdf')
plt.show()
print('fig_hake.png guardada.')


## §10 · Likert survey

In [ ]:
# ── Likert Descriptives ─────────────────────────────────────
lik_rows = []
for q in Q_COLS:
    s  = df[q]
    ci = pg.compute_bootci(s.values, func='mean', n_boot=N_BOOT,
                           confidence=.95, seed=SEED)
    lik_rows.append({
        'Item': q, 'Statement': Q_TEXT[q],
        'M': s.mean(), 'SD': s.std(ddof=1),
        'Mdn': s.median(), 'Mode': int(s.mode().iloc[0]),
        'CI95_lo': ci[0], 'CI95_hi': ci[1],
        '%≥4': (s>=4).mean()*100, '%=5': (s==5).mean()*100,
    })
lik_df = pd.DataFrame(lik_rows)

# alpha_likert / ci_likert were already calculated only once in §6;
# here they are only reused (previously it was called pg.cronbach_alpha again
# with the same data, doubling the calculation).
print(f"Cronbach's α = {alpha_likert:.3f}  IC95% [{ci_likert[0]:.3f}, {ci_likert[1]:.3f}]")
lik_df

In [ ]:
# ── Figure 7: Divergent stacked bars with seaborn pallet ─
lik_colors = sns.color_palette('RdYlBu', 5)   # diverging palette
lik_labels = ['Strongly disagree','Disagree','Neutral','Agree','Strongly agree']
freq     = np.array([[(df[q]==v).sum() for v in range(1,6)] for q in Q_COLS], dtype=float)
freq_pct = freq / len(df) * 100

fig, ax = plt.subplots(figsize=(10.5, 5))
left = np.zeros(len(Q_COLS))
for kk, (lbl, col) in enumerate(zip(lik_labels, lik_colors)):
    ax.barh(range(len(Q_COLS)), freq_pct[:,kk], left=left,
            color=col, alpha=.9, label=lbl, height=.62)
    for j in range(len(Q_COLS)):
        if freq_pct[j,kk] > 6:
            ax.text(left[j]+freq_pct[j,kk]/2, j, f'{freq_pct[j,kk]:.0f}%',
                    ha='center', va='center', fontsize=8.5, fontweight='bold',
                    color='white' if kk in (0,1,4) else '#333')
    left += freq_pct[:,kk]
for j, q in enumerate(Q_COLS):
    ax.text(102, j, f'M={df[q].mean():.2f}', va='center', fontsize=9, color='#444')
ax.set_yticks(range(len(Q_COLS)))
ax.set_yticklabels(Q_SHORT, fontsize=9.5)
ax.set_xlabel('Percentage of students (%)')
ax.set_xlim(0, 100)
ax.legend(loc='lower center', frameon=False, fontsize=8.5,
          ncol=5, bbox_to_anchor=(0.5, -0.22))
fig.tight_layout()
fig.savefig(FIG_DIR/'fig_likert_stacked.png')
fig.savefig(FIG_DIR/'fig_likert_stacked.pdf')
plt.show()
print('fig_likert_stacked.png guardada.')

## §11 Badge analysis

In [ ]:
print('Earning rates:')
for b in BADGES:
    n = df[b].sum()
    print(f'  {b:22s}: {n}/{len(df)} ({n/len(df)*100:.1f}%)')
n_all = df[BADGES].all(axis=1).sum()
print(f'  All 3 badges: {n_all}/{len(df)} ({n_all/len(df)*100:.1f}%)')

print("\npg.mwu — Hake g: got vs. didn't get every badge:")
mwu_rows = []
for b in BADGES:
    yes = df.loc[df[b],  'Hake_g']
    no_ = df.loc[~df[b], 'Hake_g']
    if len(no_) > 1:
        mwu = pg.mwu(yes, no_, alternative='two-sided')
        mwu_rows.append({
            'Badge'   : b,
            'n_yes'   : len(yes), 'g_yes': yes.mean(),
            'n_no'    : len(no_), 'g_no' : no_.mean(),
            'U'       : mwu['U_val'].iloc[0],
            'p'       : mwu['p_val'].iloc[0],
            'CLES'    : mwu['CLES'].iloc[0],
            'RBC'     : mwu['RBC'].iloc[0],
            'sig'     : stars(mwu['p_val'].iloc[0]),
        })
    else:
        mwu_rows.append({'Badge':b,'n_yes':len(yes),'g_yes':yes.mean(),
                         'n_no':0,'g_no':float('nan'),
                         'U':float('nan'),'p':float('nan'),
                         'CLES':float('nan'),'RBC':float('nan'),'sig':'—'})

pd.DataFrame(mwu_rows)

## §12 · Correlations

In [ ]:
# ── Main correlation: Hake g vs Likert ─────────────────
corr_main = pg.corr(df['Hake_g'], df['Likert_mean'], method='pearson')
print('Pearson r — Hake g vs Likert mean:')
print(corr_main.to_string())

corr_sp = pg.corr(df['Hake_g'], df['Likert_mean'], method='spearman')
print('\nSpearman rho — Hake g vs Likert mean:')
print(corr_sp.to_string())

In [ ]:
# ── Correlation matrix (pairwise) ──────────────────────
corr_vars = ['PRE','POST','Gain','Hake_g','Likert_mean','Badges_total']
corr_matrix = pg.pairwise_corr(df[corr_vars], method='pearson',
                               padjust='holm')
corr_matrix[['X','Y','n','r','CI95','p_unc','p_corr','BF10','power']]

In [ ]:
# ── Correlation heatmap ─────────────────────────────────
# r and p are reused from corr_matrix (pg.pairwise_corr, §12) instead
# rerun pearsonr() for each pair of variables.
r_mat = df[corr_vars].corr(method='pearson')

p_mat = pd.DataFrame(np.nan, index=corr_vars, columns=corr_vars)
for _, row in corr_matrix.iterrows():
    p_mat.loc[row['X'], row['Y']] = row['p_unc']
    p_mat.loc[row['Y'], row['X']] = row['p_unc']

annot = r_mat.copy().astype(str)
for a in corr_vars:
    for b in corr_vars:
        r_v = r_mat.loc[a,b]; p_v = p_mat.loc[a,b]
        annot.loc[a,b] = f'{r_v:.2f}{stars(p_v) if a!=b else ""}'

mask = np.triu(np.ones_like(r_mat, dtype=bool))
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(r_mat, mask=mask, annot=annot, fmt='', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=.5,
            cbar_kws={'shrink':.8,'label':'Pearson r'})
fig.tight_layout()
fig.savefig(FIG_DIR/'fig_corr_heatmap.png')
fig.savefig(FIG_DIR/'fig_corr_heatmap.pdf')
plt.show()

## §13 · Multiple linear regression

In [ ]:
# ── Standardize predictors (comparable betas) ─────────────
# A single StandardScaler for the entire study: PRE, Likert_mean,
# Badges_total and Hake_g. df_reg is later reused in §15
# (mediation) to avoid adjusting a second scaler on the
# same variables.
scaler = StandardScaler()
df_reg = df[['Hake_g','PRE','Likert_mean','Badges_total']].copy()
df_reg[['PRE_z','Likert_z','Badges_z','Hake_g_z']] = scaler.fit_transform(
    df_reg[['PRE','Likert_mean','Badges_total','Hake_g']]
)

# ── OLS con statsmodels ──────────────────────────────────────
model = smf.ols('Hake_g ~ PRE_z + Likert_z + Badges_z', data=df_reg).fit()
print(model.summary())

In [ ]:
# ── VIF (multicollinearity) ──────────────────────────────────
X_vif = sm.add_constant(df_reg[['PRE_z','Likert_z','Badges_z']])
vif_df = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i)
            for i in range(X_vif.shape[1])]
})
print('VIF — multicollinearity check:')
print(vif_df.to_string(index=False))

# ── Visual diagnostics with seaborn ───────────────────────
from scipy.stats import probplot as sp_probplot

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Residuals vs Fitted
fitted_df = pd.DataFrame({'Fitted': model.fittedvalues, 'Residuals': model.resid})
sns.scatterplot(data=fitted_df, x='Fitted', y='Residuals',
                ax=axes[0], color=C_HAKE, alpha=.7, s=45)
axes[0].axhline(0, color='red', ls='--', lw=1)
sns.regplot(data=fitted_df, x='Fitted', y='Residuals',
            ax=axes[0], scatter=False, color='steelblue',
            line_kws={'lw': 1.2, 'ls': ':'})
axes[0].set_title('Residuals vs Fitted')

# Q-Q residuals
(osm, osr), (slope, intercept, r) = sp_probplot(model.resid, dist='norm')
sns.scatterplot(x=osm, y=osr, ax=axes[1], color=C_PRE, alpha=0.8, s=35, zorder=3)
line_x = np.array([osm[0], osm[-1]])
axes[1].plot(line_x, slope * line_x + intercept, color=C_ACCENT, lw=1.5)
axes[1].set_xlabel('Theoretical quantiles')
axes[1].set_ylabel('Sample quantiles')
axes[1].set_title('Q-Q — residuals')

# Scale-Location
sl_df = pd.DataFrame({'Fitted': model.fittedvalues,
                       'SqrtAbsRes': np.sqrt(np.abs(model.resid))})
sns.scatterplot(data=sl_df, x='Fitted', y='SqrtAbsRes',
                ax=axes[2], color=C_POST, alpha=.7, s=45)
sns.regplot(data=sl_df, x='Fitted', y='SqrtAbsRes',
            ax=axes[2], scatter=False, color='steelblue',
            line_kws={'lw': 1.2, 'ls': ':'})
axes[2].set_ylabel('√|Residuals|')
axes[2].set_title('Scale-Location')

fig.tight_layout()
fig.savefig(FIG_DIR/'fig_regression_diagnostics.png')
fig.savefig(FIG_DIR/'fig_regression_diagnostics.pdf')
plt.show()


## §14 · Scatter Hake *g* vs. Likert satisfaction

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))

# Scatter colored by Hake g
sc = ax.scatter(df['Likert_mean'], df['Hake_g'],
                c=df['Hake_g'], cmap='rocket', s=55,
                alpha=.82, edgecolors='black', linewidths=.6, zorder=3)

# r and p reused from corr_main (pg.corr, §12) — previously called
# pg.corr() two more times in this same cell for the same thing.
r_val = corr_main['r'].iloc[0]
p_val = corr_main['p_val'].iloc[0]
sns.regplot(x=df['Likert_mean'], y=df['Hake_g'], ax=ax,
            scatter=False, color='#ad4143', line_kws={'lw':2},
            ci=95, label=f'$r = {r_val:.3f}$, $p {fmt_p(p_val)}$')

# Umbrales Hake
ax.axhline(.0, color='red', ls=':', lw=2, alpha=.5)
ax.axhline(.3, color='gold',   ls=':', lw=2, alpha=.5)
ax.axhline(.7, color='green', ls=':', lw=2, alpha=.5)
ax.text(df['Likert_mean'].min(), .71, 'High gain',
        color='green', fontsize=10, va='bottom')
ax.text(df['Likert_mean'].min(), .31, 'Medium gain',
        color='gold', fontsize=10, va='bottom')
ax.text(df['Likert_mean'].min(), .01, 'Low gain',
        color='red', fontsize=10, va='bottom')

ax.set_xlabel('Mean Likert satisfaction score (1–5)')
ax.set_ylabel('Normalized learning gain — Hake $g$')
ax.legend(frameon=False, loc='upper left')
cb = fig.colorbar(sc, ax=ax, shrink=.8, pad=.02)
cb.set_label('Hake $g$')
fig.tight_layout()
fig.savefig(FIG_DIR/'fig_scatter_hake_likert.png')
fig.savefig(FIG_DIR/'fig_scatter_hake_likert.pdf')
plt.show()
print('fig_scatter_hake_likert.png guardada.')


## §15 · Mediation analysis

In [ ]:
# Reuse the columns already standardized in df_reg (§13) instead
# to fit a second StandardScaler on the same variables.
df_med = df_reg[['Likert_z','Badges_z','Hake_g_z']].rename(
    columns={'Likert_z':'X','Badges_z':'M','Hake_g_z':'Y'}
).copy()

# Component models
outcome_model  = smf.ols('Y ~ X + M', data=df_med)
mediator_model = smf.ols('M ~ X',     data=df_med)
rng = np.random.default_rng(SEED)

med = Mediation(outcome_model, mediator_model,
                exposure='X', mediator='M')
med_result = med.fit(n_rep=N_BOOT, method='bootstrap')

print('Mediation analysis (statsmodels / Imai et al.):')
summary_df = med_result.summary()
print(summary_df)
print('\nInterpretation:')
acme_p = summary_df['P-value'].iloc[0]
print(f'  ACME (indirecto effect a·b): p = {acme_p}')
print('If 95% CI of the ACME includes 0 → NOT significant mediation.')


## §16 · Consolidated summary of results

In [ ]:
ht_global   = ht_df.loc[ht_df['Item'] == 'Global'].iloc[0]
hake_global = hake_df.loc[hake_df['Item'] == 'Global'].iloc[0]
pre_global  = desc_df.loc[(desc_df['Item'] == 'GLOBAL') & (desc_df['Phase'] == 'PRE')].iloc[0]
post_global = desc_df.loc[(desc_df['Item'] == 'GLOBAL') & (desc_df['Phase'] == 'POST')].iloc[0]

summary = {
    'N students'                : len(df),
    'N groups'                  : df['grupo'].nunique(),
    'PRE  M (SD)'               : f"{pre_global['M']:.3f} ({pre_global['SD']:.3f})",
    'POST M (SD)'               : f"{post_global['M']:.3f} ({post_global['SD']:.3f})",
    't(41)'                     : f"{ht_global['t']:.3f}",
    'p (t)'                     : '< .001',
    'Cohen d_z'                 : f"{ht_global['Cohen_dz']:.3f}",
    'Hedges g'                  : f"{ht_global['Hedges_g']:.3f}",
    'Statistical power'         : f"{ht_global['Power']:.3f}",
    'W (Wilcoxon)'              : f"{ht_global['W_wilcoxon']:.0f}",
    'CLES'                      : f"{ht_global['CLES']:.3f}",
    'Hake g global'             : f"{hake_global['g_mean']:.3f} IC95%[{hake_global['CI95_lo']:.3f},{hake_global['CI95_hi']:.3f}]",
    'Hake classification'       : hake_global['Cat'],
    'Friedman χ²'               : f"{friedman_res['Q'].iloc[0]:.3f} (p {fmt_p(friedman_res['p_unc'].iloc[0])})",
    'Kendall W'                 : f"{friedman_res['W'].iloc[0]:.3f}",
    'α Cronbach Likert'         : f"{alpha_likert:.3f} IC95%[{ci_likert[0]:.3f},{ci_likert[1]:.3f}]",
    'Likert M (SD)'             : f"{df['Likert_mean'].mean():.3f} ({df['Likert_mean'].std(ddof=1):.3f})",
    'Pearson r (g vs Likert)'   : f"{corr_main['r'].iloc[0]:.3f} (p {fmt_p(corr_main['p_val'].iloc[0])})",
    'BF10 (r)'                  : f"{corr_main['BF10'].iloc[0]}",
    'R² regresión'              : f'{model.rsquared:.3f} (adj {model.rsquared_adj:.3f})',
}

for k, v in summary.items():
    print(f'  {k:<32s}: {v}')


## §17 · Verification of generated files

In [ ]:
expected = {   
    'fig_prepost.png'               : 'Fig. 4 — PRE vs POST',
    'fig_dabest_global.png'         : 'Fig. 5a — Gardner-Altman global',
    'fig_dabest_items.png'          : 'Fig. 5b — Cumming plot ítems',
    'fig_hake.png'                  : 'Fig. 6 — Hake g',
    'fig_likert_stacked.png'        : 'Fig. 7 — Likert',
    'fig_scatter_hake_likert.png'   : 'Fig. 8 — Scatter g vs Likert',
    'fig_qqplots.png'               : 'Suppl. — Q-Q plots',
    'fig_corr_heatmap.png'          : 'Suppl. — Heatmap correlaciones',
    'fig_regression_diagnostics.png': 'Suppl. — Diagnósticos regresión',
}
print('Figure status:')
all_ok = True
for fn, desc in expected.items():
    p = FIG_DIR / fn
    ok   = p.exists()
    size = f'{p.stat().st_size/1024:.0f} KB' if ok else '—'
    print(f'  {"✓" if ok else "✗"} {fn:<40s} {size:<8s} {desc}')
    all_ok = all_ok and ok
print()
print('✓ Everything ready.' if all_ok else '✗ There are missing files.')